# Fine-tune Florence-2-base on RSVQA-LR (LoRA)

**Phase 2 of SatQuery AI.** Designed to run on a free-tier Colab T4.

Steps:
1. Install deps
2. Load Florence-2-base + processor
3. Load RSVQA-LR processed data (see `data/prepare_rsvqa.py`)
4. Wrap model with LoRA (peft)
5. Train
6. Save adapter, test a few examples
7. (Later) merge adapter or load it directly in `specialists/vqa_caption.py` with `mock=False`

In [ ]:
!pip install -q transformers accelerate peft einops timm pillow

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_ID = "microsoft/Florence-2-base"

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True, torch_dtype=torch.float16).to("cuda")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

## Load your prepared RSVQA-LR jsonl (from `data/prepare_rsvqa.py`)
Upload `train.jsonl` / `val.jsonl` and the RSVQA-LR images folder to this Colab session, or mount Drive.

In [ ]:
import json
from torch.utils.data import Dataset
from PIL import Image
import os

class RSVQADataset(Dataset):
    def __init__(self, jsonl_path, image_dir, processor):
        with open(jsonl_path) as f:
            self.examples = [json.loads(l) for l in f]
        self.image_dir = image_dir
        self.processor = processor

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        image = Image.open(os.path.join(self.image_dir, f"{ex['image_id']}.tif")).convert("RGB")
        prompt = f"<VQA> {ex['question']}"
        target = ex['answer']
        return {"image": image, "prompt": prompt, "target": target}

# train_ds = RSVQADataset('data/rsvqa_lr_processed/train.jsonl', 'raw/rsvqa_lr/images', processor)

## Attach LoRA adapters
Target the language-model attention projections (adjust `target_modules` after inspecting `model` if names differ).

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # verify against model.named_modules()
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Training loop
A minimal custom loop is shown here rather than `Trainer` since Florence-2's
processor/collation is a bit non-standard. Swap in HF `Trainer` once this is
working if you prefer.

In [ ]:
# TODO once data is loaded:
# - build a DataLoader with a custom collate_fn using `processor(...)`
# - standard train loop: forward, loss, backward, optimizer.step()
# - save with model.save_pretrained('checkpoints/florence2-rsvqa-lora')
print('Fill in once RSVQA-LR is downloaded and prepared.')

## After training
1. Save the adapter: `model.save_pretrained('checkpoints/florence2-rsvqa-lora')`
2. Download the checkpoint folder from Colab
3. In `specialists/vqa_caption.py`, set `mock=False` and point `model_path` at your checkpoint (or the base model + load the adapter with `PeftModel.from_pretrained`)